# Baseline classification for CIFAR10 dataset

In [ ]:
import sys

sys.path.append("..")

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [ ]:
SAVE_PATH = "../data"

training_dataset = datasets.CIFAR10(
    root=SAVE_PATH,
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

test_dataset = datasets.CIFAR10(
    root=SAVE_PATH,
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

In [ ]:
batch_size = 64

train_dataloader = DataLoader(training_dataset, batch_size=batch_size)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

In [ ]:
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
print(f"Using {device} device")

In [ ]:
from pytorch_helpers.neuralnetwork import NeuralNetwork  

model = NeuralNetwork(
    nn.Sequential(
        nn.Linear(3 * 32 * 32, 512),
        nn.ReLU(),
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Linear(512, 10),
    )
)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
from pytorch_helpers.training import TorchMulticlassClassifier

clf = TorchMulticlassClassifier(
    model=model,
    loss_fn=loss_fn,
    optimizer=optimizer,
    device=device,
    num_classes=10 
)

In [ ]:
clf.train(
    train_dataloader,
    max_iter=50,
    verbose=False
)

In [ ]:
import matplotlib.pyplot as plt

history = clf.plot_history()
plt.show()